# The node budget: the term we have never touched

```
0.901  submitted        0.926 bronze     0.944 gold
measurable > 0.0015      worth a slot > 0.01        (notes/44)
```

`notes/45`, read out of the official scorer rather than assumed:

```python
ADJUSTMENT_ALPHA = 0.1
J_adj = max(0, J · (1 − ADJUSTMENT_ALPHA · (N_pred − N_total) / N_total))
```

`N_pred → 0` gives a multiplier of **1.1**. Ours is **1.0012** — we predict almost exactly
`N_total`, the estimated *true* cell count, and collect none of it.

And over-prediction is otherwise free, which is why nobody noticed. `pred_valid =
out_valid | in_valid`, both taken from the GT node a prediction matched to and
`fill_null(False)` otherwise, so a predicted edge counts as TP **or** FP only if an
endpoint matched a tracked ground-truth node. Everything else is excluded, not penalised.
That is how 22,000 predicted edges against ~600 real ones still reads `edge_J = 0.935`.

Of our ~24,000 predicted nodes, roughly **670 match ground truth and ~23,300 are pure
budget cost with zero edge benefit.**

`altervation/biohub-r35-spotiflow` — a complete MIT-licensed solution from a team
plausibly at rank 35 — caps detections at **1–25 cells per frame**. We emit about 220.

## The knob, and why the config sweeps never found it

Detection in the pack is one line:

```python
is_peak = (logits == pooled) & (torch.sigmoid(logits) > det_threshold)
```

Every sweep this project ran moved `det_threshold`, and `notes/44` found that surface flat.
It is flat: thresholds from 0.965 to 0.99 change the node count by **5.6%**, so all of it
lived inside the predict-everything regime where the multiplier is pinned near 1.0 by
construction. We measured a plateau at high resolution and never stepped off it.

`pool_kernel_um` is the other half of that line — the non-maximum-suppression radius,
sitting at its default **3.0 µm**, never once swept. It controls how many peaks survive,
and node count falls roughly as its cube. That is the lever that reaches the regime the
multiplier rewards.

## What this run measures

`pool_kernel_um` ∈ {3, 6, 10, 15, 22} µm at the located `det_threshold=0.975`, on 36
datasets, with **the components reported separately**: `edge_jaccard`, `node_recall`,
`total_node_ratio`, the multiplier, and `adj_edge_jaccard`. The whole point is the shape of
the trade, not one number — the multiplier rises as `edge_J` falls, and where they cross
is the answer.

## Pre-registered predictions

1. **The anchor arm reproduces.** `pool 3.0, m6 g2` is exactly what we run today; it must
   land within 0.003 of `claude_widecv`'s 0.9348 or nothing below is comparable.
2. **Node count falls more than 5× across the grid.** `det_threshold` moved it 5.6%; if the
   kernel cannot do better, the lever does not exist and this closes cheaply.
3. **`total_node_ratio` goes below −0.5 at the largest kernel** — we actually enter the
   regime where the multiplier pays, rather than probing another plateau edge.
4. **`adj_edge_jaccard` is non-monotonic in the kernel** — it should rise, then fall, as the
   multiplier gain gives way to lost edges. Monotone down means the trade never pays;
   monotone up means the optimum is past 22 µm and the grid was too timid.
5. **The best arm beats the incumbent by more than 0.01** — `notes/44`'s bar for being worth
   a submission slot, and about three times what n=36 can resolve.

*Prediction 5 is the only outcome test here. 1 to 4 are about whether the experiment is
even looking at the thing it claims to, which is what `notes/42` said pre-registration is
actually for.*

In [ ]:
import os, subprocess, sys, time, json
from pathlib import Path

T_START = time.time()
WORK = Path("/kaggle/working"); WORK.mkdir(parents=True, exist_ok=True)

def sh(*a, **kw):
    try:
        return subprocess.run(a, capture_output=True, text=True, **kw)
    except (FileNotFoundError, OSError) as e:
        return subprocess.CompletedProcess(a, 127, "", str(e))

def pip_install(pkgs, extra=()):
    r = sh(sys.executable, "-m", "pip", "install", "-q", *extra, *pkgs)
    if r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-2000:])
    return r.returncode == 0

print(sh("nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader").stdout.strip()
      or "no GPU")

def find_dir(is_match, roots, max_depth=6):
    for root in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        stack = [(root, 0)]
        while stack:
            d, depth = stack.pop(0)
            try:
                if is_match(d):
                    return d
                if depth >= max_depth:
                    continue
                kids = [e for e in d.iterdir()
                        if e.is_dir() and e.suffix not in (".zarr", ".geff")]
            except (PermissionError, OSError):
                continue
            stack += [(k, depth + 1) for k in kids]
    return None

PACK = find_dir(lambda p: (p / "repo").is_dir() and (p / "weights").is_dir(),
                ["/kaggle/input"])
REPO = find_dir(lambda p: (p / "harness").is_dir() and (p / "pipeline").is_dir(),
                [WORK, "/kaggle/input"])
COMP = find_dir(lambda p: (p / "train").is_dir() and (p / "test").is_dir()
                and any((p / "train").glob("*.zarr")), ["/kaggle/input"])
# The torch wheelhouse: a directory of .whl files that is NOT the pack's own.
TORCH_WH = find_dir(
    lambda p: p.name == "wheels" and any(x.name.startswith("torch-") for x in p.iterdir()),
    ["/kaggle/input"])

for label, val in (("pack", PACK), ("our repo", REPO), ("competition", COMP),
                   ("torch wheels", TORCH_WH)):
    print(f"  {label:<14} {val}")
missing = [l for l, v in (("pack", PACK), ("our repo", REPO), ("competition", COMP)) if v is None]
if missing:
    raise SystemExit(f"not mounted: {missing}")
TRAIN = COMP / "train"
# Only to reuse notes/35's dataset list so the control is comparable; the candidates
# themselves are re-predicted here, not read from it.
CACHE = find_dir(lambda p: any(p.glob("cand_*.npz")), ["/kaggle/input"])
print(f"  {'cand cache':<14} {CACHE}")
# The ILP at 0.4/2.0 emits forks BY DESIGN (notes/35: div_J 0.1154), and purescore is
# only exact without them, so Harness.score_graph requires the official scorer.
CELLMOT = Path("/kaggle/working/kaggle-cell-tracking-competition")
if not (CELLMOT / "src" / "tracking_cellmot").is_dir():
    _r = sh("git", "clone", "--depth", "1",
            "https://github.com/royerlab/kaggle-cell-tracking-competition", str(CELLMOT))
    print(f"official scorer clone rc={_r.returncode}")
os.environ["CELLMOT_REPO"] = str(CELLMOT)
if not (CELLMOT / "src" / "tracking_cellmot").is_dir():
    raise SystemExit("official scorer not available; forked predictions cannot be scored")

# Offline installs, pack wheels first so numpy lands before anything compiles against it.
t0 = time.time()
ok1 = pip_install([str(p) for p in sorted((PACK / "wheels").glob("*.whl"))],
                  extra=("--no-index", f"--find-links={PACK/'wheels'}"))
print(f"pack wheels {'ok' if ok1 else 'FAILED'} ({time.time()-t0:.0f}s)")

if TORCH_WH is None:
    print("!! no torch wheelhouse attached — the P100 cannot run the image torch, so this "
          "will fall back to CPU and will NOT finish inside 12 h.")
else:
    t0 = time.time()
    ok2 = pip_install(["torch==2.5.1"], extra=("--no-index", f"--find-links={TORCH_WH}"))
    print(f"torch wheels {'ok' if ok2 else 'FAILED'} ({time.time()-t0:.0f}s)")

probe = sh(sys.executable, "-c",
           "import numpy, torch, zarr, tracksdata; "
           "ok=False\n"
           "if torch.cuda.is_available():\n"
           "    try:\n"
           "        w=torch.nn.Conv3d(1,4,3,padding=1).cuda()\n"
           "        _=w(torch.randn(2,1,8,8,8,device='cuda')).sum().item()\n"
           "        torch.cuda.synchronize(); ok=True\n"
           "    except Exception as e: print('GPU BROKEN:', type(e).__name__, str(e)[:120])\n"
           "print('numpy', numpy.__version__, '| torch', torch.__version__, '| gpu_ok', ok)")
print(probe.stdout.strip() or probe.stderr.strip()[-1500:])
if probe.returncode != 0:
    raise SystemExit("dependency stack does not import in a fresh interpreter — a scored "
                     "rerun would fail identically with no way to recover.")
if "gpu_ok True" not in probe.stdout:
    print("\n!! GPU is not usable. Continuing, but expect this to exceed the time budget; "
          "the guard below will still emit a valid file.")

## 1. Patch `predict_video`, then run the arms

`inspect.getsource` on the pack's own function, `patch_source` to insert the reverse pass,
`exec` into a **copy** of the pack module's namespace. The copy matters: mutating the pack
in place would make the control arm depend on run order.

In [ ]:
import subprocess, sys, time
WORKER = WORK / "run_bidir.py"
WORKER.write_text('''
import json, os, sys, time
from pathlib import Path
import numpy as np

os.environ["CELLMOT_REPO"] = {cellmot!r}
PACK = Path({pack!r}); REPO = Path({repo!r}); TRAIN = Path({train!r})
CACHE = {cache!r}; WORK = Path({work!r})
N_DATASETS = 36
BLEND_W = 0.15
POOL_GRID = [3.0, 6.0, 10.0, 15.0, 22.0]
DET_FIXED = 0.975
POST_GRID = [tuple(g) for g in [(6, 2), (0, 1), (6, 1), (0, 2), (8, 2), (6, 3)]]
T0 = time.time()

sys.path.insert(0, str(REPO))
sys.path.insert(0, str(PACK / "repo" / "src"))
sys.path.insert(0, str(PACK / "repo" / "scripts"))
# The pack's entry point is a SCRIPT, not a package, and it imports a `dataspec` module
# that only exists in the authors' own environment. claude_submit_ratio injects a synthetic
# one; copied from there rather than retyped, which is how v1 of this notebook came to
# import a module name that does not exist.
import types
_ds = types.ModuleType("dataspec")
_ds.USERNAME = "claude"; _ds.INTERACTIVE = False
_ds.WEIGHTS_PATH = PACK / "weights"; _ds.DATASET_PATH = TRAIN
_ds.PREDICTIONS_PATH = WORK / "predictions"
sys.modules["dataspec"] = _ds

import inspect
import torch
import tracksdata as td
from harness import Harness
from harness.tracks import Tracks, read_geff, read_scale
from harness.purescore import summarise
from pipeline.anatomy import BUCKETS, edge_anatomy, summarise_anatomy
from pipeline.repair import close_gaps, linefit_smooth, prune_short_tracks
from pipeline.bidirectional import ANCHOR, harmonic_blend, patch_source
import predict_unet_transformer as P
print("worker numpy", np.__version__, "torch", torch.__version__, flush=True)

DEV = "cuda" if torch.cuda.is_available() else "cpu"
ILP_EDGE_W, ILP_APP_W, ILP_DIS_W, ILP_DIV_W = -1.0, 0.4, 2.0, 1.0   # notes/36: the optimum
DET_THRESHOLD = 0.99

def repair_chain(g, sc):
    # A docstring here would terminate the outer f-string that writes this file.
    r = close_gaps(*g, scale=sc, max_um=5.75, max_added_frac=0.038, max_added_abs=1650)
    return linefit_smooth(*r, window=2, weight=0.76, scale=sc, max_shift_um=3.2)

ORIG_SRC = inspect.getsource(P.predict_video)
if ANCHOR not in ORIG_SRC:
    # Print the neighbourhood so the anchor can be fixed in ONE round trip rather than by
    # guessing. patch_source would catch it, but only after a GPU has been spent.
    i = ORIG_SRC.find("predict_edges")
    print("!! ANCHOR DOES NOT MATCH. predict_video source near predict_edges:", flush=True)
    print(ORIG_SRC[max(0, i - 500):i + 1000], flush=True)
    raise SystemExit("anchor mismatch -- update pipeline/bidirectional.ANCHOR")
print("anchor matches", ORIG_SRC.count(ANCHOR), "x in", len(ORIG_SRC), "chars", flush=True)

def make_predict(weight):
    # w=0 uses the ORIGINAL function object, so the control cannot differ from the
    # unpatched pipeline by even a rounding step.
    if weight == 0.0:
        return P.predict_video
    ns = dict(P.__dict__)
    exec(compile(patch_source(ORIG_SRC, weight), "<bidir>", "exec"), ns)
    return ns["predict_video"]

WPATH = PACK / "weights/unet_transformer/split_0/edge_predictor_best.pth"
model, window_size, downsample = P.load_model(WPATH, DEV)
print("model params", sum(p.numel() for p in model.parameters()),
      "window", window_size, "downsample", downsample, flush=True)
cfg = P.PredictConfig(det_threshold=DET_THRESHOLD, use_ilp=True,
                      ilp_edge_weight=ILP_EDGE_W, ilp_appearance_weight=ILP_APP_W,
                      ilp_disappearance_weight=ILP_DIS_W, ilp_division_weight=ILP_DIV_W)

# The same datasets notes/35 measured, so w=0 is comparable to 0.9179. The fallback is
# stratified -- notes/34's lesson: names[:12] sorted alphabetically gave 10 44b6 and 2
# 6bba, inverting a 71/128 population split.
seed_names = []
if CACHE:
    seed_names = sorted(p.stem[5:] for p in Path(CACHE).glob("cand_*.npz"))
    seed_names = [n for n in seed_names if (TRAIN / (n + ".geff")).exists()]
alln = sorted(p.stem for p in TRAIN.glob("*.zarr")
              if (TRAIN / (p.stem + ".geff")).exists())
print("pool", len(alln), "datasets | seed (already measured)", len(seed_names), flush=True)
rest = [n for n in alln if n not in set(seed_names)]
a = [n for n in rest if n.startswith("44b6")]
b = [n for n in rest if not n.startswith("44b6")]
need = max(0, N_DATASETS - len(seed_names))
# Proportional to the 71/128 population split, on the datasets not already taken.
k = min(len(a), max(0, round(need * len(a) / max(len(a) + len(b), 1))))
names = seed_names + a[:k] + b[:need - k]
# Stratify whatever list we ended up with. notes/34 recorded that names[:12] taken
# alphabetically inverted the embryo split; v2 of THIS notebook did it again, because the
# stratified branch only ran when the cache was missing. Slice proportionally, always.
# Deliberately NOT re-sliced here. sweep2 re-stratified after selection, which would
# drop seed datasets and break the superset property this run depends on. The selection
# above is already proportional; assert that rather than silently re-cutting it.
names = names[:N_DATASETS]
n44 = sum(n.startswith("44b6") for n in names)
print(len(names), "datasets:", n44, "x 44b6,", len(names) - n44, "x 6bba", flush=True)

h = Harness(data_dir=TRAIN, cache_dir=None)

def repair_at(g, sc, min_len, gap_max):
    # The submitted chain, with the two audited knobs exposed. min_len=0 / gap_max=1 is
    # byte-identical to what scored 0.897 -- probes/exec_config.py pins that on 30 random
    # graphs, so the anchor cell really is the current submission.
    r = close_gaps(*g, scale=sc, max_um=5.75, max_added_frac=0.038,
                   max_added_abs=1650, max_gap=gap_max)
    r = linefit_smooth(*r, window=2, weight=0.76, scale=sc, max_shift_um=3.2)
    if min_len > 0:
        r = prune_short_tracks(*r, min_frames=min_len, keep_division_components=True)
    return r

LABELS = ["p" + str(d) + "_m" + str(m) + "_g" + str(g)
          for d in POOL_GRID for m, g in POST_GRID]
ROWS = dict((l, []) for l in LABELS); ANAT = dict((l, []) for l in LABELS)
NODES = dict((l, 0) for l in LABELS); EDGES = dict((l, 0) for l in LABELS)
CAND = dict(("p" + str(d), 0) for d in POOL_GRID); PER = {{}}
pv = make_predict(BLEND_W)
print("blend", BLEND_W, "| thresholds", POOL_GRID, "|", len(POST_GRID),
      "post combos =", len(LABELS), "cells", flush=True)

BUDGET_S = 9.5 * 3600
for name in names:
    t0 = time.time()
    done = len(PER)
    if done >= 3:
        per = (time.time() - T0) / done
        if time.time() - T0 + per * 1.3 > BUDGET_S:
            print("stopping at " + str(done) + " datasets: another would cost ~"
                  + str(int(per)) + "s and the budget is " + str(int(BUDGET_S)) + "s",
                  flush=True)
            break
    sc = read_scale(TRAIN / (name + ".zarr"))
    gt = read_geff(TRAIN / (name + ".geff"))
    parts = [name]
    for det in POOL_GRID:
        # det_threshold is FIXED here; `det` is the pool kernel. Sweeping both at
        # once would confound the two halves of the same detection line.
        cfg_d = P.PredictConfig(det_threshold=DET_FIXED, pool_kernel_um=det, use_ilp=True,
                                ilp_edge_weight=ILP_EDGE_W,
                                ilp_appearance_weight=ILP_APP_W,
                                ilp_disappearance_weight=ILP_DIS_W,
                                ilp_division_weight=ILP_DIV_W)
        coords, edges = pv(model, TRAIN / (name + ".zarr"), DEV, cfg=cfg_d,
                           window_size=window_size, unet_batch_size=8,
                           downsample=downsample)
        g_td = P.build_graph(coords, edges)
        CAND["p" + str(det)] += int(g_td.num_edges())
        if g_td.num_edges():
            solver = td.solvers.ILPSolver(
                edge_weight=ILP_EDGE_W * td.EdgeAttr("edge_prob"),
                appearance_weight=ILP_APP_W, disappearance_weight=ILP_DIS_W,
                division_weight=ILP_DIV_W)
            with P.suppress_output():
                g_td = solver.solve(g_td)
        tr = Tracks.from_tracksdata(g_td)
        base = (tr.t, tr.zyx, tr.edges)
        best_here = None
        for min_len, gap_max in POST_GRID:
            lbl = "p" + str(det) + "_m" + str(min_len) + "_g" + str(gap_max)
            g = repair_at(base, sc, min_len, gap_max)
            ROWS[lbl].append(h.score_graph(name, Tracks(g[0], g[1], g[2])))
            NODES[lbl] += int(len(g[0])); EDGES[lbl] += int(len(g[2]))
            a = edge_anatomy(g[0], g[1], g[2], gt.t, gt.zyx, gt.edges, scale=sc)
            ANAT[lbl].append(a)
            if sum(a[k] for k in BUCKETS) != a["n_gt_edges"]:
                raise SystemExit(name + "/" + lbl + ": buckets do not sum")
            _r = ROWS[lbl][-1]
            v = float(_r.get("score", float("nan")))
            if v != v:
                v = float(_r.get("adj_edge_jaccard", float("nan")))
            PER.setdefault(name, {{}})[lbl] = v
            if best_here is None or v > best_here[0]:
                best_here = (v, lbl)
        parts.append("p" + str(det) + " " + format(best_here[0], ".4f"))
    print("  " + "  ".join(parts) + "   " + str(int(time.time() - t0)) + "s", flush=True)

    out = {{"arms": LABELS, "pool_grid": POOL_GRID, "det_fixed": DET_FIXED, "post_grid": [list(g) for g in POST_GRID],
           "blend_w": BLEND_W, "datasets": [n for n in names if n in PER],
           "seed_datasets": seed_names,
           "summary": dict((l, summarise(ROWS[l])) for l in LABELS if ROWS[l]),
           "anatomy": dict((l, summarise_anatomy(ANAT[l])) for l in LABELS if ANAT[l]),
           "nodes": NODES, "edges": EDGES, "candidates": CAND, "per_dataset": PER}}
    (WORK / "budget.json").write_text(json.dumps(out, indent=2, default=float))

print("worker done in", int(time.time() - T0), "s", flush=True)
'''.format(
    pack=str(PACK), repo=str(REPO), train=str(TRAIN), cache=str(CACHE or ""),
    work=str(WORK), cellmot=str(CELLMOT)))

t0 = time.time()
proc = subprocess.Popen([sys.executable, "-u", str(WORKER)],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line.rstrip(), flush=True)
rc = proc.wait()
print("worker exited", rc, "after", int(time.time() - t0), "s")
if rc != 0:
    raise SystemExit("worker failed (" + str(rc) + ")")

## 2. Five predictions, with the components reported separately

The multiplier rises as `edge_jaccard` falls, so a single number hides the mechanism.
`edge_jaccard`, `node_recall`, `total_node_ratio` and `adj_edge_jaccard` are printed apart,
and the trade is read off their shapes.

In [ ]:
import numpy as np, json, math
D = json.loads((WORK / "budget.json").read_text())
S, N, E, C = D["summary"], D["nodes"], D["edges"], D["candidates"]
ARMS, DS, PER = D["arms"], D["datasets"], D["per_dataset"]
POOLS, POST = D["pool_grid"], [tuple(g) for g in D["post_grid"]]
ANCHOR = "p3.0_m6_g2"          # the default kernel: exactly what we run today
WIDECV = 0.9348                 # claude_widecv, same chain, n=60
NN = len(PER)
EXACT = ANCHOR in S and S[ANCHOR]["score"] == S[ANCHOR]["score"]
key = "score" if EXACT else "edge_jaccard"
print(f"{NN} datasets, {len(POOLS)} pool kernels x {len(POST)} post = {len(ARMS)} cells")

def cell(p, m, g):
    return "p" + str(p) + "_m" + str(m) + "_g" + str(g)

m0, g0 = POST[0]
print(f"\n{'pool um':<10}{'score':>10}{'adj_edge':>10}{'edge_J':>9}{'node_rec':>10}"
      f"{'ratio':>9}{'mult':>8}{'nodes':>12}")
print("-" * 78)
ROWS = []
for p in POOLS:
    c = cell(p, m0, g0)
    if c not in S:
        continue
    s = S[c]
    n = N.get(c, 0)
    ratio = s.get("total_node_ratio", float("nan"))
    if ratio != ratio and s.get("edge_jaccard") and s.get("adj_edge_jaccard"):
        # derive it when the column is absent: adj = J * (1 - 0.1 * ratio)
        ratio = (1.0 - s["adj_edge_jaccard"] / s["edge_jaccard"]) / 0.1
    mult = 1.0 - 0.1 * ratio if ratio == ratio else float("nan")
    ROWS.append((p, s, n, ratio, mult))
    print(f"{p:<10}{s.get(key, float('nan')):>10.4f}"
          f"{s.get('adj_edge_jaccard', float('nan')):>10.4f}"
          f"{s.get('edge_jaccard', float('nan')):>9.4f}"
          f"{s.get('node_recall', float('nan')):>10.4f}"
          f"{ratio:>9.3f}{mult:>8.4f}{n:>12,}")

print(f"\nall {len(ARMS)} cells")
hdr = "".join(f"m{m}g{g}".rjust(11) for m, g in POST)
print(f"{'pool':<10}{hdr}")
for p in POOLS:
    row = "".join(f"{S[cell(p,m,g)][key]:>11.4f}" if cell(p, m, g) in S else " " * 11
                  for m, g in POST)
    print(f"{p:<10}{row}")

def paired(a, b):
    d = []
    for nm, r in PER.items():
        x, y = r.get(a), r.get(b)
        if x is not None and y is not None and x == x and y == y:
            d.append(x - y)
    if len(d) < 3:
        return None
    n = len(d); m = sum(d) / n
    sd = math.sqrt(sum((v - m) ** 2 for v in d) / (n - 1))
    se = sd / math.sqrt(n) if sd > 0 else 0.0
    return m, sd, se, (m / se if se else float("inf")), n

best = max((a for a in ARMS if a in S), key=lambda a: S[a][key])
inc = S.get(ANCHOR, {}).get(key, float("nan"))
print(f"\nbest {best} = {S[best][key]:.4f}   incumbent {ANCHOR} = {inc:.4f}")
print(f"\n{'arm':<16}{'mean d':>10}{'sd':>9}{'SE':>9}{'t':>8}   verdict")
for a in sorted((x for x in ARMS if x in S and x != ANCHOR), key=lambda x: -S[x][key])[:10]:
    r = paired(a, ANCHOR)
    if r is None:
        continue
    m, sd, se, t, n = r
    print(f"{a:<16}{m:>+10.4f}{sd:>9.4f}{se:>9.4f}{t:>8.2f}   "
          f"{'RESOLVED' if abs(t) > 2.0 else 'not resolved'}")

print("\n" + "=" * 92)
print("PREDICTION GRADING")
print("=" * 92)

print("\n1. the anchor arm (pool 3.0, the current default) reproduces widecv's 0.9348")
if not EXACT:
    print("   NOT GRADED — score is NaN")
else:
    ok1 = abs(inc - WIDECV) <= 0.003
    print(f"   {ANCHOR} = {inc:.4f} vs {WIDECV:.4f}  ->  {'PASS' if ok1 else 'FAIL'}")
    if not ok1:
        print("   pool_kernel_um=3.0 IS the current default, so this arm should be the")
        print("   chain we already measured. A miss means the grid is not anchored and")
        print("   nothing below compares to anything already scored. (A modest gap is")
        print("   expected from n=36 vs n=60 on a different dataset draw.)")

print("\n2. node count falls more than 5x across the kernel grid")
ns = [n for _, _, n, _, _ in ROWS if n]
if len(ns) < 2:
    print("   NOT GRADED — fewer than two arms produced counts")
else:
    fall = max(ns) / max(min(ns), 1)
    ok2 = fall > 5.0
    print(f"   {max(ns):,} -> {min(ns):,}  = {fall:.1f}x  ->  {'PASS' if ok2 else 'FAIL'}")
    if not ok2:
        print("   det_threshold moved node count 5.6% and the kernel cannot do much")
        print("   better either. The budget regime is unreachable with this detector,")
        print("   and the lever needs a detector that ranks cells, not a wider NMS.")

print("\n3. total_node_ratio goes below -0.5 at the largest kernel")
rs = [r for _, _, _, r, _ in ROWS if r == r]
if not rs:
    print("   NOT GRADED — ratio unavailable")
else:
    ok3 = min(rs) < -0.5
    print(f"   lowest ratio {min(rs):+.3f} (multiplier {1 - 0.1 * min(rs):.4f})"
          f"  ->  {'PASS' if ok3 else 'FAIL'}")
    if not ok3:
        print("   We never entered the regime the multiplier rewards, so this run")
        print("   probed another plateau edge and says nothing about the trade.")

print("\n4. adj_edge_jaccard is non-monotonic in the kernel (rises, then falls)")
adj = [s.get("adj_edge_jaccard", float("nan")) for _, s, _, _, _ in ROWS]
adj = [a for a in adj if a == a]
if len(adj) < 3:
    print("   NOT GRADED — need at least three arms")
else:
    i = int(np.argmax(adj))
    ok4 = 0 < i < len(adj) - 1
    print("   adj by kernel: " + "  ".join(f"{a:.4f}" for a in adj))
    print(f"   peak at index {i} of {len(adj)-1}  ->  {'PASS' if ok4 else 'FAIL'}")
    if i == 0:
        print("   Monotone DOWN: thinning never pays, edges are lost faster than the")
        print("   multiplier gains. The budget direction closes here.")
    elif i == len(adj) - 1:
        print("   Monotone UP: the optimum is past 22 um and the grid was too timid.")
        print("   Rerun with a wider grid before concluding anything about the size.")

print("\n5. the best arm beats the incumbent by more than 0.01 (notes/44's bar)")
r5 = paired(best, ANCHOR) if best != ANCHOR else None
if not EXACT:
    print("   NOT GRADED — score is NaN")
elif r5 is None:
    print(f"   best IS the incumbent ({ANCHOR})  ->  FAIL")
    print("   The default kernel is already optimal and the budget term is not")
    print("   reachable by non-maximum suppression.")
else:
    m, sd, se, t, n = r5
    ok5 = m > 0.01
    print(f"   {best} - {ANCHOR} = {m:+.4f}  SE {se:.4f}  t {t:.2f}  n {n}"
          f"  ->  {'PASS' if ok5 else 'FAIL'}")
    print(f"   (n={n} resolves ~{2 * sd / math.sqrt(n):.4f})")

print("\n" + "=" * 92)
print(f"n={NN}  |  best {best} {S[best][key]:.4f}  |  incumbent {ANCHOR} {inc:.4f}")
print("=" * 92)